# 🥇 Gold Layer: Business Analytics & ML Insights

This notebook demonstrates the **Gold Layer**, where data is aggregated and enriched for business consumption. It includes:
1. **Daily Sales Performance**: Aggregated metrics for dashboards.
2. **Customer Segmentation**: ML-driven clustering (Recency, Frequency, Monetary).
3. **Product Recommendations**: AI-generated cross-sell suggestions.
4. **Churn & CLV Predictions**: Advanced customer retention metrics.
5. **Next Purchase Intervals**: Timing optimization for marketing.

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, round, sum as _sum, count, countDistinct

# Production Credentials
PACKAGES = (
    "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,"
    "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,"
    "software.amazon.awssdk:bundle:2.17.178,"
    "software.amazon.awssdk:url-connection-client:2.17.178,"
    "org.apache.hadoop:hadoop-aws:3.3.1"
)

conf = (pyspark.SparkConf()
    .setAppName('Gold-Layer-Analytics')
    .set('spark.jars.packages', PACKAGES)
    .set('spark.sql.extensions', 
         'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,'
         'org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
    .set('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog')
    .set('spark.sql.catalog.nessie.uri', 'http://140.238.224.207:19120/api/v1')
    .set('spark.sql.catalog.nessie.ref', 'main')
    .set('spark.sql.catalog.nessie.authentication.type', 'NONE')
    .set('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog')
    .set('spark.sql.catalog.nessie.warehouse', 's3a://lakehouse-prod/warehouse')
    .set('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .set('spark.sql.catalog.nessie.s3.endpoint', 'https://bmcfe6z38foz.compat.objectstorage.ap-mumbai-1.oraclecloud.com')
    .set('spark.hadoop.fs.s3a.access.key', '962c9f862226831e4edea90cfcfafb8a8dffcd51')
    .set('spark.hadoop.fs.s3a.secret.key', 'sd2rGU918DTmn35E4xJ8EV7BX2XUt7DkqC8v6WDNDUw=')
    .set('spark.hadoop.fs.s3a.endpoint', 'https://bmcfe6z38foz.compat.objectstorage.ap-mumbai-1.oraclecloud.com')
    .set('spark.hadoop.fs.s3a.path.style.access', 'true')
    .set('spark.hadoop.fs.s3a.connection.ssl.enabled', 'true')
    .set('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem'))

spark = SparkSession.builder.config(conf=conf).getOrCreate()
print("✅ Spark Connected to Gold Layer")

## 📈 1. Daily Sales Performance
This table powers the Executive Dashboard. (Aggregated from Silver)

In [ ]:
# Create Sales Performance View from Silver
orders = spark.table("nessie.ecommerce.orders_silver")

sales = orders.filter(col("event_type") == "purchase") \
    .groupBy("order_date") \
    .agg(
        _sum("price").alias("total_revenue"),
        countDistinct("user_session").alias("total_orders"),
        count("product_id").alias("items_sold")
    )

print(f"Total Days Tracked: {sales.count()}")

# Show Top Revenue Days
sales.orderBy(desc("total_revenue")).show(5)

## 👥 2. Customer Segmentation (Real ML Output)
Customers are clustered (KMeans) using aggregated history from 300M events.
**Source Table**: `nessie.ecommerce.customer_segments_ml`

In [ ]:
segments = spark.table("nessie.ecommerce.customer_segments_ml")
print(f"Total Segmented Customers: {segments.count()}")

# Distribution of Segments (Column is 'cluster')
segments.groupBy("cluster").count().orderBy(desc("count")).show()

## 🔮 3. Customer Retention (Churn & CLV)
Predicting *High Risk* customers and their *Future Value* using Gradient Boosted Trees.
**Source Tables**:
- `nessie.ecommerce.churn_predictions_ml`
- `nessie.ecommerce.clv_predictions_ml`

In [ ]:
churn = spark.table("nessie.ecommerce.churn_predictions_ml")
clv = spark.table("nessie.ecommerce.clv_predictions_ml")

# Join Churn & CLV for a Unified View
retention_view = churn.join(clv, "customer_id")

# identify High Value + High Risk customers (Priority for Intervention)
high_risk_vip = retention_view.filter(
    (col("churn_probability") > 0.7) & 
    (col("predicted_clv_12m") > 1000)
)

print("⚠️ High Value (>$1000) Customers at Risk of Churn:")
high_risk_vip.orderBy(desc("predicted_clv_12m")).show(5)

## 📅 4. Next Purchase Prediction
Knowing *exactly when* a customer is likely to buy next allows for timed marketing.
**Source Table**: `nessie.ecommerce.next_purchase_predictions_ml`

In [ ]:
npp = spark.table("nessie.ecommerce.next_purchase_predictions_ml")

# Show customers likely to buy in the next 7 days
imminent_buyers = npp.filter(col("predicted_interval").between(0, 7))
print("🛒 Customers likely to buy this week:")
imminent_buyers.orderBy("predicted_interval").show(5)

## 🤖 5. Product Recommendations
Market Basket Analysis (FPGrowth) results for cross-selling.
**Source Table**: `nessie.ecommerce.product_recommendations_ml`

In [ ]:
recs = spark.table("nessie.ecommerce.product_recommendations_ml")
recs.orderBy(desc("confidence")).show(5, truncate=False)